In [2]:
import os, json, time, shutil, subprocess, threading, re, platform, glob
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import psutil
from google.colab import drive

os.environ["TZ"] = "UTC"; time.tzset()

def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":")[1].strip()
    except Exception:
        pass
    return platform.processor()

HW = {"cpu_count": os.cpu_count(), "cpu_model": cpu_model(),
      "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
      "os": platform.platform(), "python": platform.python_version()}
print("Colab hardware:\n", json.dumps(HW, indent=2))

drive.mount('/content/drive')
ROOT  = "/content/drive/MyDrive/BDC_Fraud"
CFG   = json.load(open(f"{ROOT}/config.json"))
LOCAL = "/content/bdc_local"; os.makedirs(LOCAL, exist_ok=True)
RESULTS = {}

print("\nShared rules from config.json:")
print(json.dumps(CFG["rules"], indent=2))

Colab hardware:
 {
  "cpu_count": 2,
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "ram_gb": 13.6,
  "os": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.13.15"
}
Mounted at /content/drive

Shared rules from config.json:
{
  "zscore": {
    "threshold": 3.0,
    "min_history": 10
  },
  "burst": {
    "max_txns": 3,
    "window_sec": 120
  },
  "travel": {
    "min_km": 500,
    "max_minutes": 30
  },
  "odd_hour": {
    "start_hour": 0,
    "end_hour": 5,
    "min_amount": 1500
  }
}


In [3]:
def sh(cmd, check=True, show=True, cwd=None):
    """Run a bash command, print its output, raise a clear error if it fails."""
    p = subprocess.run(cmd, shell=True, executable="/bin/bash", cwd=cwd,
                       capture_output=True, text=True, env=os.environ)
    if show and p.stdout.strip():
        print(p.stdout.strip()[-3000:])
    if check and p.returncode != 0:
        print("STDERR:", p.stderr[-3000:])
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd[:100]}")
    return p

t = time.time()
sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y openjdk-8-jdk-headless", show=False)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
sh("java -version 2>&1")
print(f"Java ready in {time.time()-t:.0f}s")

openjdk version "1.8.0_504"
OpenJDK Runtime Environment (build 1.8.0_504-8u504-ga-1ubuntu1~24.04.3-b01)
OpenJDK 64-Bit Server VM (build 25.504-b01, mixed mode)
Java ready in 15s


In [ ]:
BD = "/content/bd"; os.makedirs(BD, exist_ok=True)
HADOOP_VER, HIVE_VER = "3.3.6", "3.1.3"
URLS = {
    "hadoop": f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VER}/hadoop-{HADOOP_VER}.tar.gz",
    "hive":   f"https://archive.apache.org/dist/hive/hive-{HIVE_VER}/apache-hive-{HIVE_VER}-bin.tar.gz",
}
for name, url in URLS.items():
    t = time.time(); tgz = f"{BD}/{name}.tgz"
    if not os.path.exists(tgz):
        print(f"Downloading {name} ...")
        sh(f"wget -q -O {tgz} {url}", show=False)
    sh(f"tar -xzf {tgz} -C {BD}", show=False)
    print(f"{name}: ready ({os.path.getsize(tgz)/1e6:.0f} MB, {time.time()-t:.0f}s)")

HADOOP_HOME = f"{BD}/hadoop-{HADOOP_VER}"
HIVE_HOME   = f"{BD}/apache-hive-{HIVE_VER}-bin"
assert os.path.isdir(HADOOP_HOME) and os.path.isdir(HIVE_HOME), "extraction failed"
print("HADOOP_HOME =", HADOOP_HOME, "\nHIVE_HOME   =", HIVE_HOME)

In [ ]:
os.environ.update({
    "HADOOP_HOME": HADOOP_HOME, "HIVE_HOME": HIVE_HOME,
    "HADOOP_CONF_DIR": f"{HADOOP_HOME}/etc/hadoop",
    "HDFS_NAMENODE_USER": "root", "HDFS_DATANODE_USER": "root",
    "HDFS_SECONDARYNAMENODE_USER": "root",
    "HADOOP_OPTS": "-Djava.net.preferIPv4Stack=true",
    "HADOOP_CLIENT_OPTS": "-Xmx5g -Duser.timezone=UTC",
})
os.environ["PATH"] = ":".join([f"{HADOOP_HOME}/bin", f"{HADOOP_HOME}/sbin", f"{HIVE_HOME}/bin", os.environ["PATH"]])

def write_xml(path, props):
    body = "".join(f"  <property><name>{k}</name><value>{v}</value></property>\n" for k, v in props.items())
    open(path, "w").write(f'<?xml version="1.0"?>\n<configuration>\n{body}</configuration>\n')

conf = f"{HADOOP_HOME}/etc/hadoop"
write_xml(f"{conf}/core-site.xml", {
    "fs.defaultFS": "hdfs://localhost:9000",
    "hadoop.tmp.dir": "/content/hdfs/tmp"})
write_xml(f"{conf}/hdfs-site.xml", {
    "dfs.replication": "1",
    "dfs.namenode.name.dir": "file:///content/hdfs/name",
    "dfs.datanode.data.dir": "file:///content/hdfs/data",
    "dfs.permissions.enabled": "false"})
write_xml(f"{conf}/mapred-site.xml", {
    "mapreduce.framework.name": "local"})
with open(f"{conf}/hadoop-env.sh", "a") as f:
    f.write(f"\nexport JAVA_HOME={os.environ['JAVA_HOME']}\n")

sh("hadoop version | head -1")
print("Hadoop configured: HDFS pseudo-distributed, MapReduce local mode")

In [ ]:
sh("hdfs --daemon stop datanode; hdfs --daemon stop namenode", check=False, show=False)   # clean any leftovers
shutil.rmtree("/content/hdfs", ignore_errors=True)
sh("hdfs namenode -format -force -nonInteractive", show=False)
sh("hdfs --daemon start namenode")
sh("hdfs --daemon start datanode")

rep = ""
for _ in range(40):
    rep = sh("hdfs dfsadmin -report 2>&1 | grep 'Live datanodes'", check=False, show=False).stdout
    if "(1)" in rep: break
    time.sleep(2)
if "(1)" not in rep:
    print(sh(f"tail -n 30 {HADOOP_HOME}/logs/*namenode*.log", check=False, show=False).stdout)
    raise RuntimeError("DataNode did not register - see NameNode log above")

sh("hdfs dfsadmin -safemode wait")
sh("hdfs dfs -mkdir -p /tmp/hive /user/hive/warehouse /bdc/raw /bdc/out && hdfs dfs -chmod -R 777 /tmp /user /bdc")
print(rep.strip())
print(sh("jps", show=False).stdout)

In [ ]:
os.makedirs("/content/hive/scratch", exist_ok=True); os.makedirs("/content/hive/logs", exist_ok=True)

write_xml(f"{HIVE_HOME}/conf/hive-site.xml", {
    "javax.jdo.option.ConnectionURL": "jdbc:derby:;databaseName=/content/hive/metastore_db;create=true",
    "javax.jdo.option.ConnectionDriverName": "org.apache.derby.jdbc.EmbeddedDriver",
    "hive.metastore.warehouse.dir": "/user/hive/warehouse",
    "hive.metastore.schema.verification": "false",
    "hive.metastore.event.db.notification.api.auth": "false",
    "hive.exec.scratchdir": "/tmp/hive",
    "hive.exec.local.scratchdir": "/content/hive/scratch",
    "hive.querylog.location": "/content/hive/logs",
    "hive.server2.enable.doAs": "false",
    "hive.execution.engine": "mr"})

for g in glob.glob(f"{HIVE_HOME}/lib/guava-*.jar"):
    os.remove(g); print("removed", os.path.basename(g))
newer = glob.glob(f"{HADOOP_HOME}/share/hadoop/common/lib/guava-*.jar")[0]
shutil.copy(newer, f"{HIVE_HOME}/lib/"); print("copied ", os.path.basename(newer))

shutil.rmtree("/content/hive/metastore_db", ignore_errors=True)
out = sh("schematool -dbType derby -initSchema", cwd="/content/hive", show=False)
print("schematool:", "OK" if "completed" in out.stdout.lower() else out.stdout[-800:])

In [ ]:
sh("hdfs dfs -ls /")
t = time.time()
p = sh('hive -e "SHOW DATABASES; SHOW TABLES;"', cwd="/content/hive", check=False, show=False)
if p.returncode != 0:
    print(p.stderr[-3000:]); raise RuntimeError("Hive failed to start - use the optional PySpark fallback (Cell 14)")
print(p.stdout.strip())
print(f"Hive CLI works (startup + 2 commands: {time.time()-t:.0f}s)")

In [ ]:
def rules_select(table):
    r = CFG["rules"]; z, b, t, o = r["zscore"], r["burst"], r["travel"], r["odd_hour"]
    W = "OVER (PARTITION BY b.user_id ORDER BY b.ts, b.txn_id)"
    hav = ("2*6371.0*asin(sqrt(pow(sin(radians(lat-plat)/2),2) + "
           "cos(radians(plat))*cos(radians(lat))*pow(sin(radians(lon-plon)/2),2)))")
    return f"""WITH base AS (
  SELECT txn_id, user_id, amount, lat, lon, event_time, unix_timestamp(event_time) AS ts
  FROM {table}
),
ustats AS (
  SELECT user_id, AVG(amount) AS mu, STDDEV_SAMP(amount) AS sd, COUNT(*) AS n
  FROM base GROUP BY user_id
),
feat AS (
  SELECT b.txn_id, b.amount, b.lat, b.lon, b.event_time, b.ts, s.mu, s.sd, s.n,
         LAG(b.ts, {b['max_txns']}) {W} AS ts_nback,
         LAG(b.ts)  {W} AS pts,
         LAG(b.lat) {W} AS plat,
         LAG(b.lon) {W} AS plon
  FROM base b JOIN ustats s ON b.user_id = s.user_id
),
scored AS (
  SELECT txn_id, event_time,
    (n >= {z['min_history']} AND amount > mu + {z['threshold']} * sd)                          AS f_z,
    (ts_nback IS NOT NULL AND ts - ts_nback <= {b['window_sec']})                              AS f_b,
    (pts IS NOT NULL AND ts - pts <= {t['max_minutes'] * 60} AND {hav} > {t['min_km']})        AS f_t,
    (hour(event_time) >= {o['start_hour']} AND hour(event_time) < {o['end_hour']}
       AND amount >= {o['min_amount']})                                                        AS f_o
  FROM feat
)
SELECT txn_id,
       concat_ws('|', IF(f_z,'zscore',CAST(NULL AS STRING)), IF(f_b,'burst',CAST(NULL AS STRING)),
                      IF(f_t,'travel',CAST(NULL AS STRING)), IF(f_o,'odd_hour',CAST(NULL AS STRING))) AS reasons
FROM scored
WHERE f_z OR f_b OR f_t OR f_o"""

def create_table_hql(size):
    return f"""DROP TABLE IF EXISTS txns_{size};
CREATE EXTERNAL TABLE txns_{size} (
  txn_id STRING, user_id STRING, amount DOUBLE, merchant STRING, merchant_category STRING,
  city STRING, lat DOUBLE, lon DOUBLE, event_time STRING, is_fraud INT, fraud_type STRING)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/bdc/raw/{size}'
TBLPROPERTIES ('skip.header.line.count'='1');
SELECT COUNT(*) FROM txns_{size};"""

def rules_hql(size):
    return f"""SET mapreduce.framework.name=local;
SET hive.auto.convert.join=false;
SET mapreduce.local.map.tasks.maximum=2;
SET mapreduce.local.reduce.tasks.maximum=2;
DROP TABLE IF EXISTS flags_{size};
CREATE TABLE flags_{size} AS
{rules_select(f"txns_{size}")};
INSERT OVERWRITE DIRECTORY '/bdc/out/{size}/flags'
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
SELECT txn_id, reasons FROM flags_{size};"""

print(rules_select("txns_<size>"))

In [ ]:
class ResourceMonitor:
    """Samples system-wide CPU % and used memory every 0.5 s (Hadoop/Hive run as separate JVMs)."""
    def __init__(self, interval=0.5):
        self.interval, self.cpu, self.mem = interval, [], []
        self._stop = threading.Event()
    def _run(self):
        psutil.cpu_percent(interval=None)
        while not self._stop.is_set():
            time.sleep(self.interval)
            self.cpu.append(psutil.cpu_percent(interval=None))
            self.mem.append(psutil.virtual_memory().used / 1e6)
    def start(self):
        self.base_mem = psutil.virtual_memory().used / 1e6
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start()
    def stop(self):
        self._stop.set(); self._t.join()
        cpu = self.cpu or [0.0]; mem = self.mem or [self.base_mem]
        return {"cpu_avg_pct": round(float(np.mean(cpu)), 1), "cpu_peak_pct": round(float(np.max(cpu)), 1),
                "mem_avg_mb": round(float(np.mean(mem)), 0), "mem_peak_mb": round(float(np.max(mem)), 0),
                "mem_baseline_mb": round(self.base_mem, 0), "samples": len(cpu)}

def run_hive(hql, name):
    """Run a HiveQL script with `hive -f`; return (seconds, stdout)."""
    path = f"/content/hive/{name}.hql"; open(path, "w").write(hql)
    t = time.time()
    p = subprocess.run(f"hive -f {path}", shell=True, executable="/bin/bash", cwd="/content/hive",
                       capture_output=True, text=True, env=os.environ)
    dt = time.time() - t
    if p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-4000:])
        raise RuntimeError(f"Hive script '{name}' failed")
    return dt, p.stdout

def read_flags(path):
    try:
        return pd.read_csv(path, header=None, names=["txn_id", "flag_reasons"])
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=["txn_id", "flag_reasons"])

def package_result(size, engine, n_rows, phases, flags, res, t_start, t_end):
    """Add flag timestamp (= job completion), save flags locally, store everything in RESULTS."""
    flags = flags.copy()
    flags["flag_time"] = datetime.fromtimestamp(t_end, timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    path = f"{LOCAL}/batch_flags_{size}.csv"; flags.to_csv(path, index=False)
    RESULTS[size] = {"engine": engine, "rows": n_rows, "phases": phases,
                     "total_sec": t_end - t_start, "resource": res,
                     "flags_path": path, "n_flagged": len(flags)}
    print(f"\n--- {size.upper()} done ({engine}) ---")
    for k, v in phases.items(): print(f"  {k:<18}{v:9.2f} s")
    print(f"  {'TOTAL end-to-end':<18}{t_end - t_start:9.2f} s   |   flagged: {len(flags):,}")
print("helpers ready")

In [ ]:
def run_batch(size):
    csv_local = f"{LOCAL}/transactions_{size}.csv"
    if not os.path.exists(csv_local):
        shutil.copy(f"{ROOT}/data/transactions_{size}.csv", csv_local)
    n_rows = sum(1 for _ in open(csv_local)) - 1
    print(f"\n=== Batch pipeline | {size} | {n_rows:,} rows ===")

    sh(f"hdfs dfs -rm -r -f /bdc/raw/{size} /bdc/out/{size}", show=False)
    sh(f"hdfs dfs -mkdir -p /bdc/raw/{size}", show=False)

    mon = ResourceMonitor(); mon.start(); T0 = time.time()
    t = time.time()
    sh(f"hdfs dfs -put -f {csv_local} /bdc/raw/{size}/", show=False)
    t_load = time.time() - t
    print(f"  HDFS load done: {t_load:.1f}s")

    t_table, out = run_hive(create_table_hql(size), f"create_{size}")
    hive_count = int(re.findall(r"^\d+$", out, re.M)[-1])
    assert hive_count == n_rows, f"Hive sees {hive_count} rows, CSV has {n_rows}"
    print(f"  Hive table created, count check OK ({hive_count:,} rows): {t_table:.1f}s")

    t_query, _ = run_hive(rules_hql(size), f"rules_{size}")
    print(f"  Rules query done: {t_query:.1f}s")

    t = time.time()
    merged = f"{LOCAL}/flags_raw_{size}.csv"
    if os.path.exists(merged): os.remove(merged)
    sh(f"hdfs dfs -getmerge /bdc/out/{size}/flags {merged}", show=False)
    flags = read_flags(merged)
    t_collect = time.time() - t
    T1 = time.time(); res = mon.stop()

    phases = {"load_sec": t_load, "table_sec": t_table, "query_sec": t_query, "collect_sec": t_collect}
    package_result(size, "hive-mr-local", n_rows, phases, flags, res, T0, T1)

print("run_batch() defined")

In [ ]:
SIZES = ["10k", "100k", "1m"]

for s in SIZES:
    run_batch(s)

In [ ]:
# Cell 14 (OPTIONAL): PySpark-with-Hive-support fallback. Runs the SAME SQL (rules_select) instead of Hive CLI.
# Use it INSTEAD of Cells 6-8 and 13 if Hive will not start or runs out of memory.
sh("pip install -q pyspark==3.5.1", show=False)
from pyspark.sql import SparkSession

def run_batch_spark(size):
    spark = (SparkSession.builder.master("local[*]").appName("bdc-batch")
             .config("spark.driver.memory", "6g").config("spark.sql.shuffle.partitions", "16")
             .config("spark.sql.session.timeZone", "UTC")
             .config("spark.sql.warehouse.dir", "/content/spark-warehouse")
             .enableHiveSupport().getOrCreate())
    csv_local = f"{LOCAL}/transactions_{size}.csv"
    if not os.path.exists(csv_local):
        shutil.copy(f"{ROOT}/data/transactions_{size}.csv", csv_local)
    n_rows = sum(1 for _ in open(csv_local)) - 1
    print(f"\n=== Spark+Hive fallback | {size} | {n_rows:,} rows ===")

    mon = ResourceMonitor(); mon.start(); T0 = time.time()
    t = time.time()                                                    # load
    schema = ("txn_id string, user_id string, amount double, merchant string, merchant_category string, "
              "city string, lat double, lon double, event_time string, is_fraud int, fraud_type string")
    df = spark.read.csv(csv_local, header=True, schema=schema).cache()
    assert df.count() == n_rows
    df.createOrReplaceTempView(f"txns_{size}")
    t_load = time.time() - t

    t = time.time()                                                    # query + collect
    flags = spark.sql(rules_select(f"txns_{size}")).toPandas().rename(columns={"reasons": "flag_reasons"})
    t_query = time.time() - t
    T1 = time.time(); res = mon.stop()

    phases = {"load_sec": t_load, "table_sec": 0.0, "query_sec": t_query, "collect_sec": 0.0}
    package_result(size, "spark-hive-fallback", n_rows, phases, flags, res, T0, T1)
    df.unpersist()

# for s in ["10k", "100k", "1m"]:
#     run_batch_spark(s)

In [ ]:
def evaluate(size):
    r = RESULTS[size]
    gt = pd.read_csv(f"{LOCAL}/transactions_{size}.csv", usecols=["txn_id", "is_fraud", "fraud_type"])
    fl = pd.read_csv(r["flags_path"])
    gt["flagged"] = gt["txn_id"].isin(set(fl["txn_id"])).astype(int)

    tp = int(((gt.is_fraud == 1) & (gt.flagged == 1)).sum()); fp = int(((gt.is_fraud == 0) & (gt.flagged == 1)).sum())
    fn = int(((gt.is_fraud == 1) & (gt.flagged == 0)).sum()); tn = int(((gt.is_fraud == 0) & (gt.flagged == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    fpr  = fp / (fp + tn) if fp + tn else 0.0

    recall_by_type = gt[gt.is_fraud == 1].groupby("fraud_type")["flagged"].mean().round(4).to_dict()
    hits = fl["flag_reasons"].str.split("|").explode().value_counts().to_dict()
    fp_ids = set(gt.loc[(gt.is_fraud == 0) & (gt.flagged == 1), "txn_id"])
    fp_by_rule = fl[fl["txn_id"].isin(fp_ids)]["flag_reasons"].str.split("|").explode().value_counts().to_dict()

    ph, res, total = r["phases"], r["resource"], r["total_sec"]
    m = {
        "pipeline": "batch", "engine": r["engine"], "dataset": size, "rows": int(r["rows"]),
        "fraud_rows": int(gt.is_fraud.sum()), "flagged_rows": int(len(fl)),
        "load_sec": round(ph["load_sec"], 2), "table_sec": round(ph["table_sec"], 2),
        "query_sec": round(ph["query_sec"], 2), "collect_sec": round(ph["collect_sec"], 2),
        "ingestion_sec": round(ph["load_sec"] + ph["table_sec"], 2),
        "total_sec": round(total, 2),
        "throughput_rps": round(r["rows"] / total, 1),
        "throughput_query_rps": round(r["rows"] / ph["query_sec"], 1),
        "batch_latency_sec": round(total, 2),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4), "fpr": round(fpr, 6),
        "recall_by_type": recall_by_type, "flags_by_rule": hits, "false_positives_by_rule": fp_by_rule,
        "cpu_avg_pct": res["cpu_avg_pct"], "cpu_peak_pct": res["cpu_peak_pct"],
        "mem_avg_mb": res["mem_avg_mb"], "mem_peak_mb": res["mem_peak_mb"], "mem_baseline_mb": res["mem_baseline_mb"],
        "hardware": HW,
    }
    return m

SUMMARY = {}
for size in RESULTS:
    m = evaluate(size); SUMMARY[size] = m
    json.dump(m, open(f"{LOCAL}/batch_metrics_{size}.json", "w"), indent=2)
    shutil.copy(f"{LOCAL}/batch_metrics_{size}.json", f"{ROOT}/results/batch_metrics_{size}.json")
    shutil.copy(RESULTS[size]["flags_path"], f"{ROOT}/results/batch_flags_{size}.csv")

    print(f"\n==================== {size.upper()} ====================")
    print(pd.DataFrame([[m['tn'], m['fp']], [m['fn'], m['tp']]],
                       index=["actual normal", "actual fraud"], columns=["pred normal", "pred fraud"]))
    print("Recall by fraud type:", m["recall_by_type"])
    print("Flags by rule:", m["flags_by_rule"], "| false positives by rule:", m["false_positives_by_rule"])

keys = ["rows", "ingestion_sec", "query_sec", "total_sec", "throughput_rps", "batch_latency_sec",
        "precision", "recall", "f1", "fpr", "cpu_avg_pct", "cpu_peak_pct", "mem_avg_mb", "mem_peak_mb"]
print("\n\n############ BATCH SUMMARY (Hadoop + Hive) ############")
display(pd.DataFrame({s: {k: SUMMARY[s][k] for k in keys} for s in SUMMARY}))

In [ ]:
for size in RESULTS:
    for f in [f"results/batch_flags_{size}.csv", f"results/batch_metrics_{size}.json"]:
        fp = f"{ROOT}/{f}"
        print(f"{'OK ' if os.path.exists(fp) else 'MISSING'}  {f}  ({os.path.getsize(fp)/1e3:.1f} KB)")
drive.flush_and_unmount()
print("Drive flushed.")